In [5]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.nn import SAGEConv

import utils

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
class GraphSAGE(torch.nn.Module):
    #инициализация моделт, добавляем словарь размерности и типов
    def __init__(self, dims_dict, type2id, hidden_dim):
        super().__init__()
        self.dims_dict = dims_dict
        self.type2id = type2id
        self.hidden_dim = hidden_dim

        #модуль проекций для приведения векторов к единому размеру
        self.projections = nn.ModuleDict({
            str(t): nn.Linear(dim, hidden_dim) for t, dim in dims_dict.items()
        })

        self.conv1 = SAGEConv(hidden_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x_padded, edge_index, node_type):
        
        #логика проекции
        x = torch.zeros(x_padded.size(0), self.hidden_dim, device=x_padded.device)

        for t, type_id in self.type2id.items():
            mask = (node_type == type_id)
            dim = self.dims_dict[t]
            #вырезаем значимую часть вектора и проецируем
            x[mask] = self.projections[str(t)](x_padded[mask, :dim])

        #дальше стандартные свертки
        x = self.conv1(x, edge_index)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.conv2(x, edge_index)

        return x

def decode(z, edge_label_index):
    src = z[edge_label_index[0]]
    dst = z[edge_label_index[1]]
    return (src * dst).sum(dim=1)

def train(train_data):
    model.train()
    optimizer.zero_grad()

    z = model(train_data.x, train_data.edge_index, train_data.node_type)

    logits = decode(z, train_data.edge_label_index)
    loss_value = F.binary_cross_entropy_with_logits(logits, train_data.edge_label.float())

    loss_value.backward()
    optimizer.step()

    return loss_value.item()

In [9]:
df = pd.read_csv('../data/edges/clean_triples.csv')
folder = "../data/dicts/bert_embeddings"
all_features = utils.load_data.load_and_merge_embeddings(folder)

Loaded DNA: 630 entities
Loaded NucleicAmbigous: 16 entities
Loaded NucleicMixed: 23 entities
Loaded AA: 71376 entities
Loaded RNA: 749 entities
Loaded SmallMolecule: 1301937 entities


In [10]:
data, node_map, pre2id, type2id, dims_dict = utils.load_data.df_to_homo_different_emb(df, all_features)

interacts_mask = data.edge_type == pre2id['interacts_with']
similar_mask = data.edge_type == pre2id['has_similarity']

edge_index_interacts = data.edge_index[:, interacts_mask]
edge_index_similarity = data.edge_index[:, similar_mask]

# создаем временный граф
data_interacts = Data(
    x=data.x,
    edge_index=edge_index_interacts,
    node_type=data.node_type 
)
#все существующие ребра
all_positive_edges = data_interacts.edge_index
#сплит ребер interacts
transform = RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    is_undirected=True,
    add_negative_train_samples=True,
    neg_sampling_ratio=1
)

train_data, val_data, test_data = transform(data_interacts)

#обратно добавляем has_similarity
for split_data in [train_data, val_data, test_data]:
    split_data.edge_index = torch.cat([split_data.edge_index, edge_index_similarity], dim=1)


In [ ]:
model = GraphSAGE(dims_dict=dims_dict, type2id=type2id, hidden_dim=16).to(device)

train_data = train_data.to(device)
# val_data = val_data.to(device)
test_data = test_data.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

for epoch in range(1, 51):

    loss_value = train(train_data)

    if epoch % 10 == 0:
        print(
            f"Epoch {epoch:03d} | "
            f"Loss {loss_value:.4f} | "
        )

mrr, hits = utils.evaluating.ranking_metrics_filtered(model,
                                                        test_data,
                                                        all_pos_edge_index=all_positive_edges,
                                                        k_list=[1, 3, 10, 50],
                                                        N=10000,
                                                        use_sampling=True,
                                                        num_negatives=200)


print(f"MRR: {mrr:.4f}")
for k, v in hits.items():
    print(f"Hits@{k}: {v:.4f}")



OutOfMemoryError: CUDA out of memory. Tried to allocate 3.07 GiB. GPU 0 has a total capacity of 7.61 GiB of which 687.88 MiB is free. Including non-PyTorch memory, this process has 6.53 GiB memory in use. Of the allocated memory 6.41 GiB is allocated by PyTorch, and 21.41 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)